In [ ]:
INPUT_PBF = "../data/01_osm_pbf/noord-brabant-260914.osm.pbf"
CACHE_FOLDER = "../data/02_osm_cache/noord-brabant"

# Features are rendered in this order:
# - Earlier entries are background layers.
# - Later entries are drawn on top.
# Therefore: land cover -> water/wetlands -> roads -> buildings.

from config import CONFIG

In [ ]:
from pathlib import Path
import json

import geopandas as gpd
import pandas as pd
from pyrosm import OSM


CACHE_FOLDER = Path(CACHE_FOLDER).resolve()
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)

BOUNDARY_ADMIN_LEVEL = 4

METRIC_CRS = None

# Standard pyrosm accepts OSM(path).
#
# If your installed version supports the options from your original code,
# enable them here:
#
# OSM_OPTIONS = {
#     "engine": "out_of_core",
#     "workers": "auto",
# }
OSM_OPTIONS = {}

In [ ]:
class DataProcessor:
    def __init__(self, config, pbf_path, metric_crs=None, osm_options=None):
        self.config = config
        self.pbf_path = Path(pbf_path).resolve()

        if not self.pbf_path.is_file():
            raise FileNotFoundError(f"PBF not found: {self.pbf_path}")

        self.osm = OSM(
            str(self.pbf_path),
            **(osm_options or {}),
        )

        self.features = {}
        self.metric_crs = metric_crs
        self.boundaries = None
        self.selected_boundaries = None

    def process_features(self):
        for feature in self.config["features"]:
            name = feature["name"]
            print(f"Processing {name}...")

            gp = feature["extractFunction"](self.osm)

            if gp is None or gp.empty:
                # Filled with an empty, correctly projected GeoDataFrame below.
                self.features[name] = None
                continue

            if gp.crs is None:
                raise ValueError(f"{name}: extracted data has no CRS.")

            if self.metric_crs is None:
                self.metric_crs = gp.estimate_utm_crs()

            if self.metric_crs is None:
                raise ValueError("Could not determine a metric CRS.")

            gp = gp.to_crs(self.metric_crs)

            # Geometry functions must run AFTER reprojection so road
            # buffering is measured in metres.
            processed = feature["geometryFunction"](gp)

            if isinstance(processed, gpd.GeoSeries):
                # Preserve the original attributes when replacing geometry.
                gp = gp.copy()
                gp = gp.set_geometry(processed)
            elif isinstance(processed, gpd.GeoDataFrame):
                gp = processed
            else:
                raise TypeError(
                    f"{name}: geometryFunction must return a "
                    "GeoDataFrame or GeoSeries."
                )

            gp = gp.loc[
                gp.geometry.notna() & ~gp.geometry.is_empty
            ].copy()

            self.features[name] = gp.reset_index(drop=True)

        if self.metric_crs is None:
            raise ValueError(
                "No non-empty features were extracted. "
                "Check the PBF/configuration, or explicitly set METRIC_CRS."
            )

        for name, gp in self.features.items():
            if gp is None:
                self.features[name] = gpd.GeoDataFrame(
                    geometry=gpd.GeoSeries([], crs=self.metric_crs)
                )

    def process_boundaries(self, admin_level=4):
        print("Processing administrative boundaries...")

        boundaries = self.osm.get_boundaries(
            boundary_type="administrative"
        )

        if boundaries is None or boundaries.empty:
            raise ValueError("No administrative boundaries found in the PBF.")

        if "admin_level" not in boundaries.columns:
            raise ValueError("Administrative boundaries lack admin_level.")

        if boundaries.crs is None:
            raise ValueError("Administrative boundaries have no CRS.")

        boundaries = boundaries.to_crs(self.metric_crs)
        boundaries = boundaries.loc[
            boundaries.geometry.notna() & ~boundaries.geometry.is_empty
        ].copy()

        # Handles both strings ("4") and numeric values (4, 4.0).
        levels = pd.to_numeric(
            boundaries["admin_level"],
            errors="coerce",
        )

        selected = boundaries.loc[levels == admin_level].copy()

        if selected.empty:
            raise ValueError(
                f"No administrative boundaries found for level {admin_level}."
            )

        self.boundaries = boundaries.reset_index(drop=True)
        self.selected_boundaries = selected.reset_index(drop=True)

    def print_summary(self):
        print(f"\nMetric CRS: {self.metric_crs}")

        for name, gp in self.features.items():
            print(f"{name}: {len(gp):,}")

        if self.boundaries is not None:
            print(f"All administrative boundaries: {len(self.boundaries):,}")
            print(f"Selected boundaries: {len(self.selected_boundaries):,}")


dp = DataProcessor(
    CONFIG,
    INPUT_PBF,
    metric_crs=METRIC_CRS,
    osm_options=OSM_OPTIONS,
)

dp.process_features()
dp.process_boundaries(BOUNDARY_ADMIN_LEVEL)
dp.print_summary()

In [ ]:
import json
import numpy as np
import pandas as pd


def parquet_safe_value(value):
    """Convert arbitrary OSM-style values into Parquet-safe scalar strings."""
    if value is None or value is pd.NA:
        return None

    # JSON-encode compound OSM tag values.
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(
            value,
            ensure_ascii=False,
            default=str,
        )

    # Preserve missing floating values as nulls.
    if isinstance(value, float) and np.isnan(value):
        return None

    # Object columns may contain a mixture of int/float/str.
    # Converting all of them to string avoids Arrow schema conflicts.
    return str(value)


def save_geoparquet(gp, path):
    gp = gp.copy()

    geometry_column = gp.geometry.name

    # Parquet/Arrow requires each column to have one stable data type.
    # OSM data can contain mixed types in fields such as version, user,
    # timestamp, changeset, tags, etc.
    for column in gp.columns:
        if column == geometry_column:
            continue

        if pd.api.types.is_object_dtype(gp[column]):
            gp[column] = gp[column].map(
                parquet_safe_value
            ).astype("string")

    gp.to_parquet(
        path,
        index=False,
    )

def save_dp_to_cache(dp, cache_folder, admin_level):
    cache_folder = Path(cache_folder)
    cache_folder.mkdir(parents=True, exist_ok=True)

    manifest_path = cache_folder / "manifest.json"

    # The manifest acts as a completion marker. If this save fails midway,
    # the tile renderer will refuse to use the incomplete cache.
    manifest_path.unlink(missing_ok=True)

    feature_settings = []

    for feature in dp.config["features"]:
        name = feature["name"]
        filename = f"{name}.parquet"

        print(f"Saving {filename}...")

        save_geoparquet(
            dp.features[name],
            cache_folder / filename,
        )

        feature_settings.append({
            "name": name,
            "file": filename,
            "color": list(feature["color"]),
            "boundaryColor": (
                list(feature["boundaryColor"])
                if feature["boundaryColor"] is not None
                else None
            ),
            "tag": feature["tag"],
        })

    save_geoparquet(
        dp.boundaries,
        cache_folder / "administrative_boundaries.parquet",
    )

    selected_filename = f"boundaries_admin_{admin_level}.parquet"
    save_geoparquet(
        dp.selected_boundaries,
        cache_folder / selected_filename,
    )

    manifest = {
        "cache_version": 1,
        "source_pbf": str(dp.pbf_path),
        "metric_crs": dp.selected_boundaries.crs.to_wkt(),
        "backgroundColor": list(dp.config["backgroundColor"]),
        "defaultTag": dp.config["defaultTag"],
        "features": feature_settings,
        "boundaries": {
            "all_file": "administrative_boundaries.parquet",
            "selected_file": selected_filename,
            "admin_level": admin_level,
            "bounds": [
                float(value)
                for value in dp.selected_boundaries.total_bounds
            ],
            "tile_rule": "within_one_boundary",
        },
    }

    temporary_manifest = manifest_path.with_suffix(".json.tmp")
    temporary_manifest.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )
    temporary_manifest.replace(manifest_path)

    print(f"\nCache ready: {cache_folder}")


save_dp_to_cache(
    dp,
    CACHE_FOLDER,
    BOUNDARY_ADMIN_LEVEL,
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))

dp.selected_boundaries.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1,
)

ax.set_title(f"Administrative boundaries — level {BOUNDARY_ADMIN_LEVEL}")
plt.show()